In [ ]:
# Library set up
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys
import os
import joblib

# Lấy đường dẫn thư mục gốc (customer-churn-predictor)
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from sklearn.pipeline import Pipeline
from src.features.engineer import create_preprocessor, FeatureEngineer


#### **1. Loading Raw Data**

In [ ]:
df = pd.read_csv("../data/raw/train.csv")
df.head()

#### **2. Dropping Unnecessary Features and Missing Values**

The Dropping features are the results of EDA part, ensuring model efficiency and simplicty

In [ ]:
print('Data shape before Dropping', df.shape)
df = df.drop(
    ["CustomerID", "Tenure", "Usage Frequency", "Subscription Type", "Contract Length"], axis=1
)

df = df.dropna()
print('Data shape after Dropping', df.shape)
df.head()

#### **3. Ensuring Data Type**

In [ ]:
# int 
df['Churn'] = df['Churn'].astype(int)
df['Age'] = df['Age'].astype(int)
df['Support Calls'] = df['Support Calls'].astype(int) # Number of calls
df['Payment Delay'] = df['Payment Delay'].astype(int) # Days 
df['Last Interaction'] = df['Last Interaction'].astype(int) # Days ago

# float
df['Total Spend'] = df['Total Spend'].astype(float) # Currency Units

# Category
df['Gender'] = df['Gender'].astype('category')

df.info()

#### **4. OHE Gender**

Transform `Gender` into Binary feature `Male`, where `Male == 0` means customer is Female and `Male == 1` means otherwise


In [ ]:
gender_ohe = pd.get_dummies(df["Gender"], dtype=np.int8)
df = df.drop(["Gender"], axis=1)
df = df.join(gender_ohe)
df = df.drop(columns= 'Female')
df.head()

#### **5. Binning Numerical Features**

#### **5.1. Age Groups**

As we discuss in EDA Part: 
- Group Aldult (25 - 39 years old) has the lowest churn rate, while Senior (> 60 years old) has 100% churn rate
- The two remaining group Young Aldult (18-24) and Mid-Career (40-59) has similar churn rate (~ 55%)

Thus we will binning Age into groups as follows:

In [ ]:
def classify_age_group(age):
    if 18 <= age <= 24:
        return "Young Adult"
    elif 24 < age <= 39:
        return "Adult"
    elif 39 < age <= 59:
        return "Mid-Career"
    elif age >= 60:
        return "Senior"


df["Age Group"] = df["Age"].apply(classify_age_group)
df.head()

In [ ]:
df["Age Group"].value_counts()

In [ ]:
sns.set_palette("pastel")

sns.countplot(data=df, x="Age Group")
plt.show()

#### **5.2. Last Interaction**

To better capture customer behavior, we transformed the continuous variable Last Interaction into a categorical feature using a Binning technique. This helps the model identify non-linear patterns and risk thresholds more effectively.

Logic Breakdown:
- Highly Active (0 - 7 days): Customers who interacted within the last week. This group represents the most engaged users with the lowest churn risk.

- Active (8 - 15 days): Customers who haven't interacted in over a week but stayed within a 15-day window. They are stable but may require re-engagement.

- Dormant (> 15 days): Customers with no activity for more than two weeks. This is the high-risk zone where users are most likely to churn.

In [ ]:
def classify_interaction_frequency(last_interaction):
    if 0 < last_interaction <= 7:
        return "Highly Active"
    elif 7 < last_interaction <= 15:
        return "Active"
    elif last_interaction > 15:
        return "Dormant"


df["Interaction Frequency"] = df["Last Interaction"].apply(
    classify_interaction_frequency
)
df.head()

In [ ]:
df['Interaction Frequency'].value_counts(dropna=False)

In [ ]:
sns.countplot(data=df, x="Interaction Frequency")
plt.show()

#### **5.3. Mapping Age Groups and Interaction Frequency**

This step ensuring the datatype of new binning features are suitable for model input 

In [ ]:
age_mapping = {"Young Adult": 0, "Adult": 1, "Mid-Career": 2, "Senior": 3}

interaction_mapping = {"Highly Active": 0, "Active": 1, "Dormant": 2}

df["Age Group"] = df["Age Group"].apply(lambda age: age_mapping[age])
df["Interaction Frequency"] = df["Interaction Frequency"].apply(
    lambda age: interaction_mapping[age]
)
df.head()

#### **6. Scaler (fit for Train dataset only)**

In [ ]:
# when traning model, we will split X,y and Train, test first
# and apply scale for X_train only
scaler = StandardScaler()
df_no_label = df.drop(columns=["Churn",'Male', 'Age Group', 'Interaction Frequency'])
df_no_label = pd.DataFrame(
    scaler.fit_transform(df_no_label), columns=df_no_label.columns
)
df_no_label.head()

In [ ]:
df = df_no_label.join(df[['Male', 'Age Group', 'Interaction Frequency','Churn']])
df.head()

#### **7. Rerunning with Pipeline**

In [ ]:
# 1. raw data
df = pd.read_csv("../data/raw/train.csv")

target = "Churn"

X = df.drop(columns=[target])
y = df[target]

# 2. train, test split and Shuffle
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# 3. Preprocessing + Feature engineering Pipeline
preprocesing_pipeline = Pipeline([
   ("feature_engineering", FeatureEngineer()),
   ("preprocessor", create_preprocessor())
])

X_train_transformed = preprocesing_pipeline.fit_transform(X_train)

In [ ]:
feature_names = preprocesing_pipeline.named_steps["preprocessor"].get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_train_df.head()